# Chapter 6: Quantum Phase Estimation

## What you'll learn

- The `iterative` QPE strategy and its settings; `qiskit_standard` for circuit export and inspection
- Three time-evolution builders and their step-term trade-offs
- How to configure executors and circuit mappers
- Running IQPE end-to-end: reading `QpeResult`, alias branches, iteration circuit depths
- How QPE error scales with the number of phase bits

<a href="https://arxiv.org/abs/quant-ph/9511026" target="_blank">Quantum Phase Estimation</a> (QPE) is the canonical quantum algorithm for computing molecular ground-state energies. It measures the phase of the eigenvalue $e^{-i H t}$ accumulated under time evolution $U = e^{-i H t}$ — directly extracting the energy $E$ from the qubit Hamiltonian built in Chapter 4.


The full QPE pipeline requires four components:

| Component | Registry key | Role |
|---|---|---|
| `PhaseEstimation` | `"phase_estimation"` | Algorithm (IQPE or standard QPE) |
| `TimeEvolutionBuilder` | `"time_evolution_builder"` | Decomposes H into native gates |
| `ControlledEvolutionCircuitMapper` | `"controlled_evolution_circuit_mapper"` | Wraps evolution as controlled-U |
| `CircuitExecutor` | `"circuit_executor"` | Runs circuits on simulator or hardware |

In [ ]:
# Setup: SCF → valence → MP2 → autoCAS-EOS → qubit Hamiltonian
from pathlib import Path
import numpy as np

import qdk_chemistry.plugins.pyscf
import qdk_chemistry.plugins.qiskit

from qdk_chemistry.data import Structure
from qdk_chemistry.algorithms import available, create, print_settings
from qdk_chemistry.utils import Logger, compute_valence_space_parameters

Logger.set_global_level(Logger.LogLevel.off)

N2_XYZ = Path("../examples/data/stretched_n2.structure.xyz")
structure = Structure.from_xyz_file(N2_XYZ)

E_hf, wfn_hf = create("scf_solver").run(structure, charge=0, spin_multiplicity=1, basis_or_guess="cc-pvdz")
num_val_e, num_val_o = compute_valence_space_parameters(wfn_hf, charge=0)
wfn_val = create("active_space_selector", "qdk_valence",
                 num_active_electrons=num_val_e, num_active_orbitals=num_val_o).run(wfn_hf)
val_idx = wfn_val.get_orbitals().get_active_space_indices()
wfn_mp2 = create("orbital_localizer", "qdk_mp2_natural_orbitals").run(wfn_val, *val_idx)

loc_ham = create("hamiltonian_constructor").run(wfn_mp2.get_orbitals())
macis = create("multi_configuration_calculator", "macis_asci",
               calculate_one_rdm=True, calculate_two_rdm=True)
macis.settings().set("core_selection_strategy", "fixed")
n_a, n_b = wfn_mp2.get_active_num_electrons()
_, wfn_sci = macis.run(loc_ham, n_a, n_b)
wfn_eos = create("active_space_selector", "qdk_autocas_eos").run(wfn_sci)

# Build qubit Hamiltonian (autoCAS-EOS active space, Jordan-Wigner)
active_ham = create("hamiltonian_constructor").run(wfn_eos.get_orbitals())
qham = create("qubit_mapper", "qdk", encoding="jordan-wigner").run(active_ham)

# Exact classical reference via sparse diagonalization
solver = create("qubit_hamiltonian_solver", "qdk_sparse_matrix_solver")
E_exact, _ = solver.run(qham)

# Evolution time: T_max = π / ‖H‖₁
T_max = np.pi / qham.schatten_norm

# State preparation for the EOS wavefunction
circ_eos = create("state_prep", "sparse_isometry_gf2x").run(wfn_eos)
qc_eos = circ_eos.get_qiskit_circuit()

print(f"Qubit Hamiltonian: {qham.num_qubits} qubits, {len(qham.pauli_strings)} Pauli terms")
print(f"Schatten norm: {qham.schatten_norm:.4f}  →  T_max = {T_max:.4f}")
print(f"E_exact (FCI/EOS): {E_exact:.6f} Hartree  ← QPE target")
print(f"circ_eos: {qc_eos.num_qubits} qubits, depth={qc_eos.depth()}")


## Part 1: QPE Algorithms

Two phase estimation strategies are available:

- **`iterative`** (IQPE) — uses 1 ancilla qubit and runs `num_bits` sequential single-qubit measurements. Each iteration applies the controlled unitary to a progressively lower power and reads one bit of the phase. `shots_per_bit` controls majority-vote reliability per bit. Low qubit overhead makes this the practical choice for simulation and near-term hardware.
- **`qiskit_standard`** — textbook QPE with `n` ancilla qubits measured simultaneously after an inverse QFT. Produces a single exportable full circuit (QASM/QIR), useful for circuit inspection and QASM/QIR export. Requires `n_ancilla + n_system` qubits. `qft_do_swaps` toggles the final swap layer. For Hamiltonians of the size used in this course, the resulting circuit is too deep for near-term hardware;
  `qiskit_standard` is included here for completeness and circuit inspection.

The key trade-off: IQPE uses fewer qubits at the cost of sequential classical post-processing; standard QPE parallelises measurement into one deep circuit at the cost of ancilla overhead. **For the Hamiltonians used in this course, `iterative` is the only practical choice.**

Unlike standalone IQPE implementations in other libraries (Qiskit, PennyLane), the `iterative` component here plugs directly into the full chemistry pipeline — the same Hamiltonian built in Chapter 4 flows into QPE without any manual wiring between steps.

In [ ]:
# Inspect phase estimation algorithms and settings
print("Available phase estimators:", available("phase_estimation"))
print()
for name in available("phase_estimation"):
    print(f"--- {name} ---")
    print_settings("phase_estimation", name)
    print()

# Two QPE strategies:
#   iterative       — IQPE: 1 ancilla qubit; num_bits sequential single-qubit measurements.
#                     Each iteration k applies controlled-U^(2^(n-k-1)), then measures.
#                     shots_per_bit: majority vote per bit (≥10 for reliable phase readout).
#                     Low qubit overhead; the practical choice for near-term simulation.
#   qiskit_standard — Textbook QPE: n ancilla qubits + inverse QFT at the end.
#                     Produces a single exportable full circuit (QASM/QIR for hardware).
#                     qft_do_swaps: toggles the final swap layer in the inverse QFT.
#                     Requires n_ancilla + n_system qubits; deeper but parallelised.


## Part 2: Time Evolution Builders

The time evolution step decomposes <code>U = e<sup>−iHt</sup></code> into native 2-qubit gate sequences. Three strategies are available, each producing a `TimeEvolutionUnitary` container with a list of `step_terms`:

- **`trotter`** — Suzuki-Trotter product formula. Order 1 makes one forward pass through all Pauli terms; order 2 adds a symmetric backward pass, halving the Trotter error at twice the step count. Error is deterministic and analyzable via commutator bounds. See <a href="https://cloudblogs.microsoft.com/quantum/2023/06/08/microsoft-quantum-researchers-make-algorithmic-advances-to-tackle-intractable-problems-in-physics-and-materials-science/" target="_blank">Microsoft Research (2023)</a> for recent advances in recursive Trotter steps that further reduce implementation cost.
- **`qdrift`** — randomly samples Pauli terms weighted by |h_j|. Produces fewer step terms for large Hamiltonians; error scales with λ²t²/N rather than with term count alone.
- **`partially_randomized`** — hybrid: deterministic Trotter for heavy Pauli terms, qDRIFT sampling for light ones. Reduces both step count and systematic Trotter error relative to pure Trotter.

In [ ]:
# Inspect time evolution builder settings
print("Available time evolution builders:", available("time_evolution_builder"))
print()
for name in available("time_evolution_builder"):
    print(f"--- {name} ---")
    print_settings("time_evolution_builder", name)
    print()

# Three builder families:
#   trotter             — Suzuki-Trotter product formula. Order 1 = Lie-Trotter;
#                         order 2 = symmetric Strang splitting (lower error).
#                         Deterministic — error fully analyzable via commutator bounds.
#   qdrift              — Randomly samples Pauli terms weighted by |h_j|. Fewer step terms
#                         for large Hamiltonians; error ∝ λ²t²/N not t²/N.
#   partially_randomized — Deterministic Trotter for heavy terms, qDRIFT for light ones.
#                          Hybrid: reduces both step count and systematic Trotter error.


## Part 3: Builder Comparison

`step_terms` maps directly to 2-qubit gate blocks in each IQPE iteration circuit — fewer terms means shallower circuits and less gate noise on hardware.

On the autoCAS-EOS Hamiltonian (161 Pauli terms): Trotter order 1 = 161 steps; Trotter order 2 = 321 (symmetric forward + backward); qDRIFT = 70 (random subset); partially randomized = 124 (deterministic core + random light terms). qDRIFT wins on raw depth but adds shot-to-shot variance. For a fixed `shots_per_bit` budget, partially randomized often gives the best error-vs-depth trade-off.

In [ ]:
# Compare time evolution builders on step term count
print(f"{'Builder':<28} {'Step terms':>12}")
print("-" * 43)
for label, name, params in [
    ("trotter (order=1)",      "trotter",              {"order": 1}),
    ("trotter (order=2)",      "trotter",              {"order": 2}),
    ("qdrift",                 "qdrift",               {"num_samples": 100, "seed": 42}),
    ("partially_randomized",   "partially_randomized", {"seed": 42}),
]:
    teb = create("time_evolution_builder", name)
    for k, v in params.items():
        teb.settings().set(k, v)
    tev = teb.run(qham, T_max)
    c   = tev.get_container()
    print(f"{label:<28} {len(c.step_terms):>12}")

print()
print("Trotter order=1: one pass through all 161 Pauli terms = 161 step terms.")
print("Trotter order=2: symmetric product (forward + reversed) = 2×161 = 321.")
print("qDRIFT: random subset — only 70 sampled terms; unbiased but adds variance.")
print("Partially randomized: 124 — deterministic core reduces systematic Trotter error.")
# Each step term → a 2-qubit gate block in the IQPE iteration circuit.
# Fewer step terms → shallower circuits → less gate noise on hardware.


# Question: qpe-builder-cp897z

> Question not found (HTTP 404)


## Part 4: Executors and Circuit Mapper

The `CircuitExecutor` runs each iteration circuit on a backend. Three executors are available:

- **`qdk_sparse_state_simulator`** — sparse statevector; efficient for ≤~20 qubits. The default choice for this course.
- **`qdk_full_state_simulator`** — dense statevector; builds the full 2ⁿ vector in memory. Only practical for <14 qubits.
- **`qiskit_aer_simulator`** — Qiskit Aer backend; supports `QuantumErrorProfile` noise models for realistic hardware noise simulation.

The only available circuit mapper is `pauli_sequence`, which expresses each exp(−iθP/2) term as a CNOT ladder + Rz rotation wrapped with a control qubit.

In [ ]:
# Circuit executor and controlled circuit mapper settings
print("Available circuit executors:", available("circuit_executor"))
print()
for name in available("circuit_executor"):
    print(f"--- {name} ---")
    print_settings("circuit_executor", name)
    print()

print("Available controlled evolution circuit mappers:", available("controlled_evolution_circuit_mapper"))
print()
for name in available("controlled_evolution_circuit_mapper"):
    print(f"--- {name} ---")
    print_settings("controlled_evolution_circuit_mapper", name)
    print()

# Three circuit executors:
#   qdk_sparse_state_simulator — Sparse statevector; efficient for ≤~20 qubits.
#                                 Handles IQPE ancilla+system circuits without full 2^n RAM.
#   qdk_full_state_simulator   — Dense statevector; builds the full 2^n × 1 vector.
#                                 Practical only for very small qubit counts (< 14 qubits).
#   qiskit_aer_simulator       — Qiskit Aer backend; supports QuantumErrorProfile noise models
#                                 for realistic hardware noise simulation. Requires qiskit plugin.
# One mapper: pauli_sequence — expresses each exp(-i θ/2 P) as a CNOT ladder + Rz rotation,
#             wrapped with a control qubit. The only available mapper; power=1 for single-step.


## Part 5: Running IQPE

IQPE runs `num_bits` sequential circuits. Iteration k applies <code>U<sup>2<sup>(n−k−1)</sup></sup></code> — the first iteration is the deepest; each subsequent one halves the depth.

`pe.run()` returns a `QpeResult` with three key fields:
- `raw_energy` — the energy selected from the phase measurement (Hartree)
- `bitstring_msb_first` — the measured binary fraction, most-significant bit first
- `branching` — alias candidates: QPE measures phase modulo 2π, so multiple energies separated by 2π/T are equally consistent with the measurement. Pass a classical reference energy (e.g., from CCSD) to resolve the correct alias when the result is ambiguous.

Fill in the executor name in the cell below, and then answer the subsequent question.

In [ ]:
# Run iterative QPE and interpret QpeResult
teb = create("time_evolution_builder", "trotter")
teb.settings().set("order", 1)
mapper   = create("controlled_evolution_circuit_mapper", "pauli_sequence")
executor = create("circuit_executor", "???")  # fill in: "qdk_sparse_state_simulator" or "qdk_full_state_simulator"
executor.settings().set("seed", 42)

pe = create("phase_estimation", "iterative")
pe.settings().set("evolution_time", T_max)
pe.settings().set("num_bits", 8)
pe.settings().set("shots_per_bit", 10)

result = pe.run(circ_eos, qham,
                evolution_builder=teb, circuit_mapper=mapper, circuit_executor=executor)

print(result.get_summary())
print(f"\nE_exact:    {E_exact:.6f} Hartree")
print(f"raw_energy: {result.raw_energy:.6f} Hartree  (error: {(result.raw_energy - E_exact)*1000:.1f} mHa)")
print()
print("Alias branches (2π/T periodicity, 5 candidates):")
for b in result.branching:
    marker = " ← selected" if abs(b - result.raw_energy) < 1e-4 else ""
    print(f"  {b:>10.4f} Hartree{marker}")
print()
print("QPE measures the phase φ of eigenvalue e^{-iEt}; energy E = -φ/T.")
print(f"2π/T_max ≈ {2*np.pi/T_max:.2f} Hartree — alias spacing.")
print("For ambiguous cases, pass reference_energy to QpeResult.from_phase_fraction()")
print("to select the alias closest to a classical reference (e.g., CCSD energy).")
print()
# Iteration circuit anatomy: IQPE uses 1 ancilla + n_system qubits.
# Iteration k applies controlled-U^(2^(num_bits-k-1)); depths halve each step.
print("IQPE iteration circuit depths (9 qubits = 1 ancilla + 8 system):")
print(f"{'Iteration':<12} {'Depth':>10} {'CX gates':>10}")
print("-" * 36)
for i, circ in enumerate(pe._iteration_circuits[:4]):
    qc = circ.get_qiskit_circuit()
    ops = qc.count_ops()
    print(f"{i+1:<12} {qc.depth():>10} {ops.get('cx', 0):>10}")
print("  ... (each iteration halves the depth)")
print()
print("Iteration 1 applies U^128 (= U^(2^7)): deepest circuit, dominates hardware cost.")
print("Total circuit depth ∝ step_terms × 2^num_bits — direct link back to Ch.4 Schatten norm.")


# Free Response Question: chem-iqpe-error-l64i2i

What is the error you get in the cell above, between the exact energy and raw energy?

## Expected Answer

['-10.5']


## Part 6: Accuracy vs Phase Bits

QPE energy resolution is $\Delta \varepsilon = \frac{2\pi}{T \cdot 2^n}$. Each added bit halves the grid spacing. Chemical accuracy is **1.6 mHa** (= 1 kcal/mol) — the threshold below which quantum chemistry predictions are considered reliable enough to guide experiment.

The sweep shows: 4 bits → ~401 mHa; 6 bits → ~97 mHa; 8 bits → ~21 mHa; 10 bits → ~1.7 mHa (quantization error approaching chemical accuracy — Trotter and gate errors are independent and can dominate). In practice, three independent error sources stack: **Trotter error** (systematic; reduce with higher order or more steps), **shot noise** (statistical; reduce with more `shots_per_bit`), and **gate noise** (hardware decoherence; irreducible without error correction). For classical simulation, 8–10 bits with `shots_per_bit ≥ 10` is a reasonable starting point. On near-term quantum hardware, gate noise makes this many phase bits impractical without error correction.

One active research direction for reducing QPE resource requirements is basis set optimization: choosing a more compact orbital basis can reduce the Schatten norm by up to 80%, directly reducing the time parameter T — at fixed target accuracy, a smaller T requires fewer Trotter steps and correspondingly shallower circuits (<a href="https://arxiv.org/abs/2509.05733" target="_blank">Kottmann et al., 2025</a>).

In [ ]:
# QPE accuracy vs number of phase bits
print(f"{'num_bits':<10} {'bitstring':<14} {'raw_energy':>15} {'error (mHa)':>12}")
print("-" * 54)
for nb in [4, 6, 8, 10]:
    teb_i = create("time_evolution_builder", "trotter")
    teb_i.settings().set("order", 1)
    exec_i = create("circuit_executor", "qdk_sparse_state_simulator")
    exec_i.settings().set("seed", 42)
    pe_i = create("phase_estimation", "iterative")
    pe_i.settings().set("evolution_time", T_max)
    pe_i.settings().set("num_bits", nb)
    pe_i.settings().set("shots_per_bit", 10)
    res = pe_i.run(circ_eos, qham,
                   evolution_builder=teb_i,
                   circuit_mapper=create("controlled_evolution_circuit_mapper", "pauli_sequence"),
                   circuit_executor=exec_i)
    err = (res.raw_energy - E_exact) * 1000
    print(f"{nb:<10} {res.bitstring_msb_first:<14} {res.raw_energy:>15.6f} {err:>12.1f}")

print()
print("Each added bit halves the energy grid: Δε = 2π / (T · 2ⁿ).")
print("10 bits → ~1.7 mHa error, approaching chemical accuracy (1.6 mHa = 1 kcal/mol).")
print("Error sources on real hardware:")
print("  Trotter error — systematic; reducible with higher order or more steps")
print("  Shot noise    — statistical; reducible with more shots_per_bit")
print("  Gate noise    — hardware decoherence; reducible with error correction")
print()
print("→ End-to-end pipeline complete: Structure → HF → active space →")
print("  qubit Hamiltonian (Ch.4) → state prep (Ch.5) → QPE → ground state energy.")


# Multiple Choice Question: qpe-accuracy-qiyblv

QPE energy resolution scales as $\Delta \varepsilon = \frac{2\pi}{T \cdot 2^n}$. How many phase bits are needed for the QPE quantization error to approach chemical accuracy (1.6 mHa = 1 kcal/mol)?

## Choices

A. 4 bits: error ~401 mHa
B. 6 bits: error ~97 mHa
C. 8 bits: error ~21 mHa
D. 0 bits: error ~1.7 mHa


## Summary

In this chapter you:
- Inspected `iterative` QPE (the practical choice for this course) and `qiskit_standard` (for circuit export)
- Inspected three time-evolution builders (Trotter, qDRIFT, partially-randomized) and their step-term counts
- Inspected all circuit executors and the `pauli_sequence` controlled circuit mapper
- Ran IQPE end-to-end and read `QpeResult`: `raw_energy`, `bitstring_msb_first`, alias `branching`
- Inspected IQPE iteration circuit depths (simulator gate counts for the N₂ autoCAS-EOS system) — halving from ~208k to ~26k over 4 iterations
- Observed QPE error scaling: 401 mHa (4 bits) → 97 mHa (6) → 21 mHa (8) → 1.7 mHa (10)

**End-to-end pipeline complete:**
Structure (Ch.0–1) → HF/SCF (Ch.2) → active space (Ch.3) → qubit Hamiltonian (Ch.4) → state prep (Ch.5) → **QPE** → ground state energy

**Where this is heading:** The pipeline you ran here targets fault-tolerant hardware. For the first demonstration of an end-to-end QPE calculation with quantum error correction on real hardware, see <a href="https://arxiv.org/abs/2505.09133" target="_blank">Babbush et al. (2025)</a>.